# Logistic Regression for omics data to classify tumor vs. non-tumor
Author: Inika, 2026-05-06

Used to classify tumor vs. normal samples for methylation data and CNV data

# Load libraries & data

In [1]:
import pandas as pd

In [2]:
meth = pd.read_csv("data/DNAm.csv")
# meth.head

In [3]:
# Read in metadata
metadata = pd.read_csv("data/metadata.csv")
metadata

,patient_id,project,label,sample_type,has_RNASeq,has_DNAm,has_CNV
0,TCGA-AK-3458,TCGA-KIRC,1,Primary Tumor,True,False,True
1,TCGA-B0-5711,TCGA-KIRC,1,Primary Tumor,True,False,True
2,TCGA-B0-5696,TCGA-KIRC,1,Primary Tumor,True,False,True
3,TCGA-CJ-4882,TCGA-KIRC,1,Primary Tumor,True,False,True
4,TCGA-B0-5109,TCGA-KIRC,1,Primary Tumor,True,False,True
...,...,...,...,...,...,...,...
775,TCGA-B0-4713-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
776,TCGA-B0-5094-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
777,TCGA-BP-5198-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False
778,TCGA-B0-5100-11A,TCGA-KIRC,0,Solid Tissue Normal,False,True,False


In [4]:
df = meth

In [5]:
# Use sample names as rownames in metadata
metadata = metadata.set_index("patient_id", drop=False)

# Reformat sample names
df["sample_id"] = df["sample_id"].str.replace(".", "-", regex=False)

def shorten_tcga_id(x):
    
    parts = x.split("-")
    # TCGA barcode structure: sample-level ends at element 4
    # if there are more parts → strip last two
    if len(parts) > 4:
        return "-".join(parts[:4])
    return x


df["sample_id"] = df["sample_id"].apply(shorten_tcga_id)

In [6]:
df.columns

Index(['sample_id', 'cg21870274', 'cg00168193', 'cg08258224', 'cg16619049',
       'cg18147296', 'cg13938959', 'cg12445832', 'cg23999112', 'cg11527153',
       ...
       'cg08219170', 'cg16997375', 'cg11598976', 'cg25005368', 'cg02953382',
       'cg01238044', 'cg01957799', 'cg20064778', 'cg24238852', 'cg11478607'],
      dtype='object', length=384630)

In [7]:
metadata.columns

Index(['patient_id', 'project', 'label', 'sample_type', 'has_RNASeq',
       'has_DNAm', 'has_CNV'],
      dtype='object')

In [8]:
# Subset metadata to only include samples available for this omics
metadata_bool = metadata["patient_id"].isin(df["sample_id"])
metadata_short = metadata[metadata_bool]

In [9]:
print(df.shape)
print(metadata_short.shape)

(446, 384630)
(446, 7)


# Use pre-selected genes

In [10]:
topgenes_multiomics = pd.read_csv("results/top_genes_multiomics.csv", sep = ";")
topgenes_multiomics

,GEX,METH,CNV
0,MIF,cg23097686,FAM174A
1,DGCR5,cg04456219,ST8SIA4
2,EEF1G,cg02326386,SACM1L
3,AQP2,cg01702055,SLCO4C1
4,UBD,cg11201447,SLC25A46
...,...,...,...
495,HMGCR,cg03584506,CCL5
496,ATP6V1A,cg06613738,RDM1
497,MTHFS,cg05471495,MMP28
498,HINT2,cg24390590,GAS2L2


In [11]:
cnv_genes = topgenes_multiomics["METH"].tolist()
cols = ["sample_id"] + list(df.columns.intersection(cnv_genes))
df = df[cols]
df


,sample_id,cg09364122,cg01961086,cg13434489,cg15819225,cg25755851,cg25282951,cg14780070,cg11336311,cg09993145,...,cg24587601,cg14303122,cg17534029,cg13309012,cg00717080,cg09476092,cg07529658,cg27115863,cg18072282,cg21737444
0,TCGA-BP-4760-01A,0.932000,0.223000,0.816000,0.796000,0.269000,0.249000,0.369000,0.593,0.830000,...,0.731000,0.781000,0.221000,0.362000,0.942,0.159000,0.391000,0.058000,0.535000,0.067000
1,TCGA-A3-A6NI-01A,0.401000,0.171000,0.750000,0.245000,0.140000,0.153000,0.273000,0.092,0.650000,...,0.059000,0.226000,0.032000,0.285000,0.404,0.314000,0.832000,0.053000,0.164000,0.110000
2,TCGA-BP-5010-01A,0.619000,0.594000,0.590000,0.394000,0.131000,0.358000,0.334000,0.286,0.581000,...,0.095000,0.124000,0.046000,0.341000,0.632,0.330000,0.987000,0.100000,0.256000,0.169000
3,TCGA-CJ-5684-01A,0.469000,0.547000,0.680000,0.351000,0.152000,0.264000,0.278000,0.263,0.538000,...,0.118000,0.339000,0.260000,0.322000,0.477,0.273000,0.897000,0.081000,0.183000,0.176000
4,TCGA-B8-A54G-01A,0.577000,0.591000,0.508000,0.336000,0.098000,0.264000,0.278000,0.555,0.402000,...,0.090000,0.099000,0.044000,0.044000,0.545,0.527000,0.972000,0.099000,0.038000,0.167000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
441,TCGA-B0-4713-11A,0.963201,0.130266,0.889032,0.893338,0.531601,0.885000,0.816707,0.787,0.909025,...,0.744500,0.713000,0.506357,0.822580,0.956,0.792268,0.409052,0.481724,0.564000,0.358996
442,TCGA-B0-5094-11A,0.966241,0.205458,0.911686,0.869828,0.537448,0.885000,0.690800,0.787,0.871827,...,0.744500,0.713000,0.407013,0.737659,0.956,0.773752,0.596543,0.412759,0.484767,0.442605
443,TCGA-BP-5198-11A,0.960833,0.269960,0.849018,0.861115,0.641846,0.640445,0.687569,0.787,0.903527,...,0.300129,0.386407,0.505035,0.735500,0.956,0.808415,0.524293,0.396059,0.434277,0.499923
444,TCGA-B0-5100-11A,0.965843,0.392661,0.922693,0.789955,0.331233,0.885000,0.730286,0.787,0.803581,...,0.744500,0.713000,0.470287,0.632961,0.956,0.865099,0.605511,0.374396,0.464013,0.471606


# Split test & training data

In [12]:
# Split training & test data
from sklearn.model_selection import train_test_split

X = df.drop(columns=["sample_id"])
y = metadata_short["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# Logistic regression for Methylation data

In [13]:
# Do feature selection and set up the pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("clf", LogisticRegression(
        solver = "saga",
        penalty="elasticnet",
        l1_ratio = 0.75,
        C = 0.75,
        max_iter=100,
        class_weight="balanced"
    ))
])

In [14]:
# Train the model
pipeline.fit(X_train, y_train)

/root/miniforge3/envs/SysCompBio_2026_env/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/root/miniforge3/envs/SysCompBio_2026_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'elasticnet'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.75
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.75
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=True

In [15]:
model = pipeline.named_steps["clf"]

importance = pd.Series(
   model.coef_[0],
   index=X_train.columns
).sort_values(key=abs, ascending=False)
print(importance)


cg04571941   -0.865677
cg23097686   -0.654381
cg22137572   -0.574693
cg23892310    0.428773
cg02104138   -0.407752
                ...   
cg09597022    0.000000
cg09365002    0.000000
cg07156249    0.000000
cg02525705    0.000000
cg21737444    0.000000
Length: 500, dtype: float64


In [16]:
# Evaluate the model
from sklearn.metrics import classification_report, roc_auc_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

              precision    recall  f1-score   support

           0       0.80      0.88      0.84        32
           1       0.93      0.88      0.90        58

    accuracy                           0.88        90
   macro avg       0.86      0.88      0.87        90
weighted avg       0.88      0.88      0.88        90

ROC-AUC: 0.9089439655172413
